# Generate Counterfactual Explanations

This notebook generates DiCE counterfactual explanations for a set of pre-selected cases. For each case the algorithm finds a small set of feature changes, drawn from the 5th-95th percentile range of the training distribution, that would flip the predicted outcome from default to no default.


In [10]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import dice_ml
from pathlib import Path

# Configuration
DATA_DIR = Path("../data")
MODELS_DIR = Path("../models")
RESULTS_DIR = Path("../results")
TARGET_COL = "status"
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [11]:
class CreditMLP(nn.Module):
    def __init__(self, input_dim):
        super(CreditMLP, self).__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.1),

            nn.Linear(32, 1)
            # Sigmoid removed: model was trained with BCEWithLogitsLoss.
            # Previous: nn.Sigmoid()
        )

    def forward(self, x):
        return self.network(x)


class PyTorchModelWrapper:
    """Wrapper to make PyTorch model compatible with DiCE"""

    def __init__(self, pytorch_model, device='cpu'):
        self.model = pytorch_model
        self.device = device
        self.model.eval()

    def predict_proba(self, X):
        if isinstance(X, pd.DataFrame):
            X = X.values

        X_tensor = torch.FloatTensor(X).to(self.device)

        with torch.no_grad():
            logits = self.model(X_tensor)
            proba_1 = torch.sigmoid(logits).cpu().numpy().flatten()

        proba_0 = 1 - proba_1
        return np.column_stack([proba_0, proba_1])

    def predict(self, X):
        proba = self.predict_proba(X)
        return (proba[:, 1] >= 0.5).astype(int)

## 1. Load Trained Model

In [12]:
checkpoint = torch.load(MODELS_DIR / 'mlp_model.pth', map_location=DEVICE, weights_only=False)
input_dim = checkpoint['input_dim']

model = CreditMLP(input_dim).to(DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

model_wrapper = PyTorchModelWrapper(model, DEVICE)

print(f"Model loaded: {sum(p.numel() for p in model.parameters()):,} parameters")

Model loaded: 18,113 parameters


## 2. Load Data

In [13]:
train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

print(f"Train: {train_df.shape}, Test: {test_df.shape}")

Train: (118936, 57), Test: (14867, 57)


## 3. Load Pre-Selected High-Risk Cases

In [14]:
selected_cases_df = pd.read_csv(DATA_DIR / "high_risk_cases.csv")
case_ids = selected_cases_df['case_id'].tolist()

print(f"Selected {len(case_ids)} cases: {case_ids}")
selected_cases_df[['case_id', 'predicted_probability', 'predicted_label', 'status']]

Selected 10 cases: [6421, 3028, 12412, 2509, 7794, 14308, 9761, 10984, 9264, 3542]


,case_id,predicted_probability,predicted_label,status
0,6421,0.918707,1,1
1,3028,0.918673,1,1
2,12412,0.916750,1,0
3,2509,0.871348,1,1
4,7794,0.862572,1,0
5,14308,0.794301,1,1
6,9761,0.701655,1,1
7,10984,0.693526,1,0
8,9264,0.675350,1,1
9,3542,0.669001,1,0


## DiCE Configuration

The explainer is configured with the mutable features only. Demographic variables, credit_score, and income are held fixed.


In [15]:
# Mutable features - only features applicants can control during loan application
MUTABLE_FEATURES = ['loan_amount', 'property_value', 'ltv', 'term', 'dtir1']

# Continuous features for DiCE (all non-one-hot-encoded features)
CONTINUOUS_FEATURES = [
    'term', 'credit_score', 'ltv', 'dtir1', 'loan_amount',
    'income', 'property_value', 'year', 'loan_limit_cf', 'loan_limit_ncf'
]

# Setup DiCE explainer
dice_data = dice_ml.Data(
    dataframe=train_df,
    continuous_features=CONTINUOUS_FEATURES,
    outcome_name=TARGET_COL
)

dice_model = dice_ml.Model(
    model=model_wrapper,
    backend='sklearn',
    model_type='classifier'
)

explainer = dice_ml.Dice(dice_data, dice_model, method='random')

print(f"DiCE explainer configured with {len(MUTABLE_FEATURES)} mutable features")

DiCE explainer configured with 5 mutable features


## Generate Counterfactuals

Five counterfactual scenarios are generated for each selected case.


In [16]:
X_test = test_df.drop(columns=[TARGET_COL])
all_results = []

for case_id in case_ids:
    query_instance = X_test.iloc[[case_id]]
    
    try:
        dice_exp = explainer.generate_counterfactuals(
            query_instance,
            total_CFs=5,
            desired_class=0,
            features_to_vary=MUTABLE_FEATURES
        )
        
        cf_df = dice_exp.cf_examples_list[0].final_cfs_df
        
        if cf_df is not None and len(cf_df) > 0:
            all_results.append({
                'case_index': case_id,
                'counterfactuals': cf_df,
                'original': query_instance,
                'success': True
            })
        else:
            all_results.append({
                'case_index': case_id,
                'counterfactuals': None,
                'success': False,
                'original': query_instance
            })
    
    except Exception as e:
        all_results.append({
            'case_index': case_id,
            'counterfactuals': None,
            'success': False,
            'error': str(e),
            'original': query_instance
        })

success_count = sum(1 for r in all_results if r['success'])
print(f"Generated counterfactuals for {success_count}/{len(case_ids)} cases")

100%|██████████| 1/1 [00:00<00:00, 14.55it/s]


No Counterfactuals found for the given configuration, perhaps try with different parameters... ; total time taken: 00 min 00 sec


100%|██████████| 1/1 [00:00<00:00,  5.98it/s]

Generated counterfactuals for 9/10 cases


## Verify Predictions

Counterfactuals are passed back through the model to confirm that the predicted class does in fact flip from default to no default.


In [17]:
verification_summary = []

for result in all_results:
    if not result['success'] or result['counterfactuals'] is None:
        continue
    
    case_idx = result['case_index']
    original = result['original']
    cf_df = result['counterfactuals']
    
    orig_proba = model_wrapper.predict_proba(original.values)[0, 1]
    orig_pred = int(orig_proba >= 0.5)
    
    cf_features = cf_df.drop(columns=[TARGET_COL], errors='ignore')
    cf_probas = model_wrapper.predict_proba(cf_features.values)[:, 1]
    cf_preds = (cf_probas >= 0.5).astype(int)
    
    flipped = [cf_pred != orig_pred for cf_pred in cf_preds]
    
    verification_summary.append({
        'case_index': case_idx,
        'original_proba': orig_proba,
        'original_pred': orig_pred,
        'num_counterfactuals': len(cf_probas),
        'num_flipped': sum(flipped),
        'flip_rate': sum(flipped) / len(flipped) if flipped else 0
    })

verification_summary = pd.DataFrame(verification_summary)

# Display results
high_risk_cases = verification_summary[verification_summary['original_pred'] == 1]
if len(high_risk_cases) > 0:
    print(f"High-risk flip rate: {high_risk_cases['flip_rate'].mean():.1%}")

verification_summary

High-risk flip rate: 100.0%


,case_index,original_proba,original_pred,num_counterfactuals,num_flipped,flip_rate
0,6421,0.997229,1,5,5,1.0
1,12412,0.995371,1,5,5,1.0
2,2509,0.881189,1,5,5,1.0
3,7794,0.863398,1,5,5,1.0
4,14308,0.803660,1,5,5,1.0
5,9761,0.680932,1,5,5,1.0
6,10984,0.622304,1,2,2,1.0
7,9264,0.632149,1,5,5,1.0
8,3542,0.548859,1,5,5,1.0


## 7. Save Results

In [18]:
dice_results_dir = RESULTS_DIR / "dice_counterfactuals"
dice_results_dir.mkdir(parents=True, exist_ok=True)

# Save verification summary
verification_summary.to_csv(dice_results_dir / "verification_summary.csv", index=False)

# Save individual counterfactuals
for result in all_results:
    if result['success'] and result['counterfactuals'] is not None:
        case_idx = result['case_index']
        cf_file = dice_results_dir / f"counterfactuals_case_{case_idx}.csv"
        result['counterfactuals'].to_csv(cf_file, index=False)

print(f"Results saved to {dice_results_dir}")

Results saved to ../results/dice_counterfactuals
